# RAG Evaluation Report

Run the RAG evaluator and view the report. Execute from **project root** or ensure the first code cell adds the project root to `sys.path`.

## Setup

In [ ]:
import sys
import json
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)

print("Project root:", ROOT)

## Run evaluator

In [ ]:
from eval.evaluator import run_eval_sync

data_path = ROOT / "eval" / "datasets" / "eval.jsonl"
run_id = datetime.utcnow().strftime("%Y%m%d_%H%M")
out_dir = ROOT / "eval" / "reports" / f"run_{run_id}"

if not data_path.exists():
    raise FileNotFoundError(f"Eval dataset not found: {data_path}")

report = run_eval_sync(str(data_path), top_k=5, judge="nli", out_dir=out_dir)
print("Report written to:", out_dir)

## View summary

In [ ]:
r = report.get("retrieval", {})
g = report.get("generation", {})
e = report.get("e2e", {})
s = report.get("system", {})

print("=== Retrieval ===")
print(f"Recall@k: {r.get('recall_at_k', 0):.4f}  MRR@k: {r.get('mrr_at_k', 0):.4f}  nDCG@k: {r.get('ndcg_at_k', 0):.4f}")
print(f"Coverage: {r.get('coverage', 0):.4f}  Redundancy: {r.get('redundancy', 0):.4f}")
print("=== Generation ===")
print(f"Faithfulness (NLI): {g.get('faithfulness_nli', 0):.4f}  Hallucination rate: {g.get('hallucination_rate', 0):.4f}")
print(f"Answer relevance: {g.get('answer_relevance', 0):.4f}  Attribution P/R: {g.get('attribution_precision', 0):.4f}/{g.get('attribution_recall', 0):.4f}")
print("=== E2E ===")
print(f"Exact match: {e.get('exact_match', 0):.4f}  F1: {e.get('f1', 0):.4f}  Nugget F1: {e.get('nugget_f1', 0):.4f}")
print("=== System ===")
print(f"Latency retrieve p50/p95 (s): {s.get('latency_retrieve_p50_s', 0):.4f} / {s.get('latency_retrieve_p95_s', 0):.4f}")
print(f"Latency generate p50/p95 (s): {s.get('latency_generate_p50_s', 0):.4f} / {s.get('latency_generate_p95_s', 0):.4f}")
print(f"Tokens in/out: {s.get('tokens_in_total', 0)} / {s.get('tokens_out_total', 0)}")